# 여행 에이전트 예제

| 정보 | 세부 내용 |
|---|---|
| 튜토리얼 | Harness - 생성, 호출, Observability, Memory |
| SDK | boto3 |
| 모델 | Claude Haiku 4.5 (Bedrock) |

**사전 요구 사항:**
- Amazon Bedrock AgentCore에 접근할 수 있는 AWS 계정
- 자격 증명이 설정된 AWS CLI v2
- `uv` 설치

## 0단계: 설정

헬퍼를 가져오고 IAM 실행 역할을 생성합니다.

In [ ]:
import sys
import time
import uuid
from pathlib import Path
import boto3

# 헬퍼
sys.path.insert(0, str(Path.cwd().parent.parent))

# --- 설정 ---
from helper.iam import create_harness_role, delete_harness_role
from helper.client import get_agentcore_control_client, get_agentcore_client

# --- boto3 클라이언트 생성 ---
control = get_agentcore_control_client()
client = get_agentcore_client()

account_id = boto3.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}")

In [ ]:
role_arn = create_harness_role()
print(f"\nExecution Role ARN: {role_arn}")

print("Waiting for IAM role to propagate...")
time.sleep(10)
print("Ready!")

## Part 1: Harness 생성(Control Plane)

AgentCore는 다음 plane으로 구성됩니다.

- **Control Plane** - 에이전트 리소스 관리(harness 생성, 읽기, 업데이트, 삭제 등)
- **Data Plane** - 에이전트 실행(호출, 응답 스트리밍, microVM 안에서 명령 실행)

이 단계에서는 Control Plane을 사용하여 Harness 리소스를 생성합니다. 명령은 `status: CREATING`과 함께 즉시 반환됩니다. 이후 `READY`로 전환될 때까지 폴링합니다.

> 💡 `HARNESS_NAME`을 원하는 이름으로 변경할 수 있습니다. 시스템이 고유성을 보장하기 위해 무작위 접미사를 추가합니다.

In [ ]:
HARNESS_NAME = f"TravelGuideAgent_{uuid.uuid4().hex[:8]}"

resp = control.create_harness(
    harnessName=HARNESS_NAME,
    executionRoleArn=role_arn,
)
harness = resp["harness"]
harness_id = harness["harnessId"]
harness_arn = harness["arn"]
print(f"Harness ID: {harness_id}")
print(f"Harness ARN: {harness_arn}")
print(f"Status: {harness['status']}")

### READY 상태 확인

Harness 프로비저닝에는 몇 초가 걸립니다. 이 셀은 상태가 `READY`가 될 때까지 5초마다 폴링합니다.

In [ ]:
for i in range(12):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        print("\u2705 Harness is ready")
        break
    time.sleep(5)

## Part 2: Harness 호출(Data Plane)

이제 boto3를 통해 Data Plane API를 사용하여 에이전트를 호출합니다. `client`는 설정 셀에서 구성했습니다.

각 호출에는 격리된 microVM을 식별하는 `session_id`가 필요합니다. 같은 세션 ID는 상태가 유지되는 같은 VM을, 새 세션 ID는 새 VM을 의미합니다.

응답은 이벤트 스트림입니다. 텍스트 델타가 도착하는 대로 출력하고 에이전트가 수행하는 도구 호출을 표시합니다.

> 💡 `modelId`를 편집하여 **모델**을 변경할 수 있습니다.
>
> 💡 **프롬프트**를 원하는 내용으로 변경할 수 있습니다. 에이전트에서는 기본적으로 `file_operations` 및 `shell` 도구를 사용할 수 있습니다.

In [ ]:
session_id = str(uuid.uuid4()).upper()
print(f"Session ID: {session_id}")

In [ ]:
response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "Recommend three fun things to do in NYC on a rainy day. Save your answer as a single self-contained HTML file at /tmp/travel.html. The HTML should have a modern dark-themed design with horizontal swipeable cards (one per activity) that the user can navigate using left/right arrow buttons. Each card should include the activity name, a short description, and a practical tip. Include all CSS and JS inline."
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

### 에이전트의 HTML 출력 렌더링

에이전트가 자체 완결형 HTML 파일을 VM에 저장했습니다. `ExecuteCommand`를 사용하여 가져온 뒤 `IPython.display.HTML`로 노트북에서 직접 렌더링합니다.

In [ ]:
from IPython.display import HTML

# 에이전트 VM에서 HTML 가져오기
html_content = ""
resp = client.invoke_agent_runtime_command(
    agentRuntimeArn=harness_arn,
    runtimeSessionId=session_id,
    body={"command": "cat /tmp/travel.html"},
)
for event in resp["stream"]:
    if "chunk" in event:
        chunk = event["chunk"]
        if "contentDelta" in chunk and "stdout" in chunk["contentDelta"]:
            html_content += chunk["contentDelta"]["stdout"]

# 노트북의 스타일과 격리하도록 iframe 안에서 렌더링
import html as html_mod

iframe = f'<iframe srcdoc="{html_mod.escape(html_content)}" width="100%" height="480" style="border:none;border-radius:12px;"></iframe>'
HTML(iframe)

## Part 3: AgentCore Observability 살펴보기

모든 Harness 호출은 AgentCore Observability를 통해 CloudWatch에 트레이스를 자동으로 생성합니다. 모델 호출, 도구 호출, 타이밍 세부 정보 등 에이전트 루프의 각 단계를 확인할 수 있습니다.

**최초 1회 설정:** CloudWatch에서 계정별로 [Transaction Search를 활성화](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/CloudWatch-Transaction-Search-getting-started.html)해야 합니다. 트레이스를 보려면 이 설정이 필요합니다.

> 💡 아래 셀은 브라우저에서 CloudWatch X-Ray 콘솔을 직접 엽니다. 최근 호출의 트레이스를 찾으면 전체 에이전트 루프의 세부 내역을 확인할 수 있습니다.

In [ ]:
import webbrowser

# Transaction Search 활성화 여부 확인
import boto3

xray = boto3.client("xray", region_name="us-west-2")
rules = xray.get_indexing_rules()
sampling = rules["IndexingRules"][0]["Rule"]["Probabilistic"]["DesiredSamplingPercentage"]
print(f"Transaction Search sampling: {sampling}%")
if sampling == 0:
    print("⚠️  Transaction Search is not enabled. Enable it in CloudWatch console first.")
else:
    print("✅ Transaction Search is enabled")

# 브라우저에서 CloudWatch X-Ray 트레이스 열기
url = "https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#xray:traces"
webbrowser.open(url)
print(f"\n🔗 Opened: {url}")

## Part 4: AgentCore Memory 추가

Memory를 Harness에 연결하면 모든 호출의 대화가 세션 ID 범위의 Memory 인스턴스에 자동으로 저장됩니다. 이후 같은 세션 ID로 호출하면 에이전트가 추론을 시작하기 전에 Memory에서 기록을 불러옵니다. 따라서 VM 세션이 만료된 뒤에도 이전 턴을 기억합니다.

이전 메시지를 직접 전달할 필요 없이 새 메시지만 보내면 에이전트가 이전 대화에 이어서 작업합니다.

**단계:**
1. Memory 인스턴스 생성(프로비저닝에 약 5분 소요)
2. Harness와 연결
3. 멀티턴 대화 테스트

> 💡 `event-expiry-duration`은 Memory 보관 기간을 일 단위로 제어합니다. 사용 사례에 맞게 조정하세요.

### 4.1 Memory 인스턴스 생성

In [ ]:
try:
    # Memory API는 beta endpoint가 아닌 표준 AgentCore endpoint를 사용
    resp = control.create_memory(
        name="TravelGuideMemory",
        eventExpiryDuration=30,
        description="Memory for TravelGuideAgent",
    )
    memory_id = resp.get("id") or resp.get("memory", {}).get("id")
except Exception as e:
    print(f"Create returned: {e}")
    memory_id = None

if not memory_id:
    print("Looking for existing TravelGuideMemory...")
    resp = control.list_memories()
    for m in resp.get("memories", []):
        if "TravelGuide" in m.get("id", ""):
            memory_id = m["id"]
            break

assert memory_id, "Could not find or create TravelGuideMemory"
print(f"Memory ID: {memory_id}")

In [ ]:
for i in range(30):
    resp = control.get_memory(memoryId=memory_id)
    status = resp.get("status", resp.get("memory", {}).get("status", "UNKNOWN"))
    print(f"Attempt {i + 1}: {status}")
    if status in ("ACTIVE", "READY"):
        print("\u2705 Memory is ready")
        break
    time.sleep(10)

In [ ]:
memory_arn = resp["memory"]["arn"]

### 4.2 Memory와 Harness 연결

In [ ]:
control.update_harness(
    harnessId=harness_id,
    memory={"optionalValue": {"agentCoreMemoryConfiguration": {"arn": memory_arn}}},
)
print(f"Memory ARN: {memory_arn}")
print("Waiting for Harness to update...")

for i in range(12):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        print("\u2705 Harness updated")
        break
    time.sleep(5)

### 4.3 Memory 테스트 - 멀티턴 대화

같은 세션에서 두 개의 메시지를 보냅니다. 두 번째 메시지는 첫 번째 메시지를 참조하며, 에이전트는 Memory 덕분에 컨텍스트를 기억해야 합니다.

In [ ]:
memory_session_id = str(uuid.uuid4()).upper()
print(f"Memory test session: {memory_session_id}\n")

# 첫 번째 대화
print("=== Turn 1 ===")
resp = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=memory_session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "My name is John Doe and I love electronic music with balance — deep house, nu-disco, anything with a nice groove. Remember that."
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
)
for event in resp["stream"]:
    if "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()

In [ ]:
# Turn 2 - 에이전트가 이름과 선호도를 기억해야 함
print("=== Turn 2 ===")
resp = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=memory_session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "What's my name and what kind of music do I like? Recommend a place in Amsterdam where I can enjoy it."
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
)
for event in resp["stream"]:
    if "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()

## Part 5: Browser 도구 - 실시간 웹 데이터

`agentcore_browser` 도구는 에이전트에 microVM 안의 실제 브라우저를 제공합니다. 웹사이트 탐색, 링크 클릭, 콘텐츠 추출 및 실시간 데이터 가져오기가 가능하므로 최신 정보가 필요한 여행 에이전트에 적합합니다.

> 💡 호출 시 `tools`를 전달하면 해당 호출에서 브라우저를 활성화할 수 있습니다. Harness 설정은 변경되지 않으며 이 호출에서만 재정의됩니다.

In [ ]:
browser_session_id = str(uuid.uuid4()).upper()
print(f"Browser session: {browser_session_id}\n")

response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=browser_session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "Check the weather forecast for Amsterdam this week. "
                    "Browse a real weather website to get current, accurate data. "
                    "Save the forecast as a clean HTML file at /tmp/weather.html with a modern dark theme, "
                    "showing each day as a card with temperature, conditions, and an emoji for the weather."
                }
            ],
        }
    ],
    tools=[{"type": "agentcore_browser", "name": "browser"}],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

### 일기 예보 렌더링

에이전트 VM에서 HTML 파일을 가져와 노트북에서 렌더링합니다.

In [ ]:
from IPython.display import HTML
import html as html_mod

html_content = ""
resp = client.invoke_agent_runtime_command(
    agentRuntimeArn=harness_arn,
    runtimeSessionId=browser_session_id,
    body={"command": "cat /tmp/weather.html"},
)
for event in resp["stream"]:
    if "chunk" in event:
        chunk = event["chunk"]
        if "contentDelta" in chunk and "stdout" in chunk["contentDelta"]:
            html_content += chunk["contentDelta"]["stdout"]

if html_content:
    iframe = f'<iframe srcdoc="{html_mod.escape(html_content)}" width="100%" height="480" style="border:none;border-radius:12px;"></iframe>'
    display(HTML(iframe))
else:
    print("No HTML file found — the agent may not have created it.")

## Part 6: Exa Search + Code Interpreter - 관광 데이터 분석

같은 세션의 여러 호출에서 **여러 도구를 조합**하는 방법을 보여 줍니다. 에이전트는 다음 작업을 수행합니다.
1. **Exa**(MCP를 통한 웹 검색)를 사용하여 실제 Amsterdam 관광 데이터 검색
2. **Code Interpreter**를 사용하여 데이터를 분석하고 차트 생성

> 💡 같은 호출에 여러 도구를 전달할 수 있습니다. 에이전트가 사용할 도구와 순서를 결정합니다.

In [ ]:
research_session_id = str(uuid.uuid4()).upper()
print(f"Research session: {research_session_id}\n")

# 1단계: Exa로 관광 데이터 검색
print("=== Step 1: Searching with Exa ===")
response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=research_session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "Search for Amsterdam tourism statistics — visitor numbers by year, "
                    "top 5 most visited attractions, and monthly visitor trends. "
                    "Collect the data and save it as a structured JSON file at /tmp/tourism_data.json."
                }
            ],
        }
    ],
    tools=[
        {
            "type": "remote_mcp",
            "name": "exa",
            "config": {"remoteMcp": {"url": "https://mcp.exa.ai/mcp"}},
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
    timeoutSeconds=300,
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

print("\n=== Step 1 complete ===")

In [ ]:
# 2단계: Code Interpreter로 차트 생성(같은 세션에서는 데이터가 유지됨)
print("=== Step 2: Generating chart with Code Interpreter ===")
response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=research_session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "Read the tourism data from /tmp/tourism_data.json and create a beautiful "
                    "visualization chart using matplotlib. Make a bar chart or line chart showing "
                    "the most interesting trends you found. Use a dark theme with vibrant colors. "
                    "Save the chart as /tmp/amsterdam_tourism.png and a brief summary as /tmp/tourism_report.md."
                }
            ],
        }
    ],
    tools=[{"type": "agentcore_code_interpreter", "name": "code_interpreter"}],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
    timeoutSeconds=300,
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

print("\n=== Step 2 complete ===")

### 생성된 보고서 및 차트 보기

In [ ]:
# 관광 보고서 읽기
report = ""
resp = client.invoke_agent_runtime_command(
    agentRuntimeArn=harness_arn,
    runtimeSessionId=research_session_id,
    body={"command": "cat /tmp/tourism_report.md 2>/dev/null || echo 'No report found'"},
)
for event in resp["stream"]:
    if "chunk" in event:
        chunk = event["chunk"]
        if "contentDelta" in chunk and "stdout" in chunk["contentDelta"]:
            report += chunk["contentDelta"]["stdout"]
print(report)

In [ ]:
# PNG로 생성된 경우 차트 표시
import base64
from IPython.display import Image, display

chart_data = b""
resp = client.invoke_agent_runtime_command(
    agentRuntimeArn=harness_arn,
    runtimeSessionId=research_session_id,
    body={"command": "base64 /tmp/amsterdam_tourism.png 2>/dev/null || echo 'NO_CHART'"},
)
for event in resp["stream"]:
    if "chunk" in event:
        chunk = event["chunk"]
        if "contentDelta" in chunk and "stdout" in chunk["contentDelta"]:
            chart_data += chunk["contentDelta"]["stdout"].encode()

chart_str = chart_data.decode().strip()
if chart_str and chart_str != "NO_CHART":
    display(Image(data=base64.b64decode(chart_str)))
    print("\u2705 Chart rendered from /tmp/amsterdam_tourism.png")
else:
    print("No chart found — the agent may have saved it in a different format or location.")
    # 생성된 항목의 목록 확인 시도
    resp = client.invoke_agent_runtime_command(
        agentRuntimeArn=harness_arn,
        runtimeSessionId=research_session_id,
        body={"command": "ls -la /tmp/*.png /tmp/*.html /tmp/*.md 2>/dev/null"},
    )
    for event in resp["stream"]:
        if "chunk" in event:
            chunk = event["chunk"]
            if "contentDelta" in chunk and "stdout" in chunk["contentDelta"]:
                print(chunk["contentDelta"]["stdout"], end="")

## Part 7: 로컬 Chat UI

이 Part에서는 위에서 생성한 Harness 에이전트에 연결되는 가벼운 채팅 웹 앱을 저장합니다.

노트북은 `travel_chat/`에 두 파일을 작성합니다.
- **`server.py`** - SSE 스트리밍으로 메시지를 `invoke_harness`에 프록시하는 FastAPI 백엔드
- **`index.html`** - 최소한의 채팅 UI

아래 셀을 실행하세요. 마지막 셀에서 서버를 시작하는 정확한 명령을 출력합니다.

In [ ]:
!mkdir -p travel_chat

In [ ]:
%%writefile travel_chat/server.py
# /// script
# requires-python = ">=3.10"
# dependencies = ["fastapi", "uvicorn", "boto3", "sse-starlette"]
# ///
import os, uuid, json, logging
from contextlib import asynccontextmanager
import boto3, botocore.session
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from sse_starlette.sse import EventSourceResponse

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

HARNESS_ARN = os.environ["HARNESS_ARN"]
REGION = os.getenv("AWS_DEFAULT_REGION")
DATA_ENDPOINT = os.getenv("DATA_ENDPOINT")

def make_client():
    kwargs = {"region_name": REGION}
    return boto3.client("bedrock-agentcore", **kwargs)

client = make_client()
sessions = {}

@asynccontextmanager
async def lifespan(app):
    logger.info("Chat app started")
    yield

app = FastAPI(lifespan=lifespan)

@app.get("/")
async def index():
    return FileResponse(os.path.join(os.path.dirname(__file__), "index.html"))

@app.post("/session")
async def new_session():
    sid = str(uuid.uuid4()).upper()
    sessions[sid] = []
    return {"session_id": sid}

@app.post("/chat")
async def chat(req: dict):
    msg = req.get("message", "").strip()
    sid = req.get("session_id", "")
    if not msg or not sid:
        raise HTTPException(400, "message and session_id required")

    if sid not in sessions:
        sessions[sid] = []
    sessions[sid].append({"role": "user", "content": [{"text": msg}]})

    async def stream():
        try:
            resp = client.invoke_harness(
                harnessArn=HARNESS_ARN,
                runtimeSessionId=sid,
                messages=sessions[sid],
            )
            full = ""
            for event in resp["stream"]:
                if "contentBlockDelta" in event:
                    txt = event["contentBlockDelta"].get("delta", {}).get("text", "")
                    if txt:
                        full += txt
                        yield {"data": json.dumps({"type": "text_delta", "text": txt})}
            if full:
                sessions[sid].append({"role": "assistant", "content": [{"text": full}]})
            yield {"data": json.dumps({"type": "done"})}
        except Exception as e:
            logger.error(str(e), exc_info=True)
            yield {"data": json.dumps({"type": "error", "message": str(e)})}

    return EventSourceResponse(stream())

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)


### 채팅 UI 저장

In [ ]:
%%writefile travel_chat/index.html
<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Travel Guide Chat</title>
<style>
  * { margin:0; padding:0; box-sizing:border-box; }
  body { font-family:system-ui; background:#1a1a2e; color:#eee; height:100vh; display:flex; flex-direction:column; }
  #header { padding:16px 24px; background:#16213e; border-bottom:1px solid #0f3460; }
  #header h1 { font-size:18px; color:#e94560; }
  #messages { flex:1; overflow-y:auto; padding:20px; }
  .msg { max-width:75%; margin:8px 0; padding:12px 16px; border-radius:12px; line-height:1.5; white-space:pre-wrap; }
  .user { background:#0f3460; margin-left:auto; }
  .assistant { background:#16213e; }
  #input-bar { display:flex; padding:16px; gap:8px; background:#16213e; border-top:1px solid #0f3460; }
  #input { flex:1; padding:12px; border-radius:8px; border:1px solid #0f3460; background:#1a1a2e; color:#eee; font-size:15px; }
  #send { padding:12px 24px; border-radius:8px; border:none; background:#e94560; color:#fff; cursor:pointer; font-size:15px; }
  #send:disabled { opacity:0.5; }
</style></head><body>
<div id="header"><h1>✈️ Travel Guide Agent</h1></div>
<div id="messages"></div>
<div id="input-bar">
  <input id="input" placeholder="Ask about any destination..." />
  <button id="send" onclick="send()">Send</button>
</div>
<script>
let sessionId = null;
const msgs = document.getElementById("messages");
const inp = document.getElementById("input");
const btn = document.getElementById("send");
inp.addEventListener("keydown", e => { if(e.key==="Enter"&&!e.shiftKey){e.preventDefault();send();} });

async function init() {
  const r = await fetch("/session",{method:"POST"});
  sessionId = (await r.json()).session_id;
}
init();

function addMsg(role, text) {
  const d = document.createElement("div");
  d.className = "msg " + role;
  d.textContent = text;
  msgs.appendChild(d);
  msgs.scrollTop = msgs.scrollHeight;
  return d;
}

async function send() {
  const text = inp.value.trim();
  if(!text) return;
  inp.value = "";
  btn.disabled = true;
  addMsg("user", text);
  const el = addMsg("assistant", "");
  try {
    const r = await fetch("/chat",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({message:text,session_id:sessionId})});
    const reader = r.body.getReader();
    const dec = new TextDecoder();
    let buf = "";
    while(true) {
      const {done, value} = await reader.read();
      if(done) break;
      buf += dec.decode(value, {stream:true});
      const lines = buf.split("\n");
      buf = lines.pop();
      for(const line of lines) {
        if(!line.startsWith("data: ")) continue;
        const d = JSON.parse(line.slice(6));
        if(d.type==="text_delta") el.textContent += d.text;
        if(d.type==="error") el.textContent += "\nError: "+d.message;
      }
      msgs.scrollTop = msgs.scrollHeight;
    }
  } catch(e) { el.textContent += "\nError: "+e; }
  btn.disabled = false;
  inp.focus();
}
</script></body></html>


## 리소스 정리

테스트를 마치면 유휴 Harness 또는 Memory 인스턴스에 대한 요금이 발생하지 않도록 **아래 셀을 실행하여 모든 리소스를 삭제**하세요.

이 노트북에서 생성한 리소스:
- **Harness** - `delete-harness`로 삭제
- **Memory 인스턴스** - `delete-memory`로 삭제
- **IAM 역할** - `delete_harness_role()`로 삭제

In [ ]:
control.delete_harness(harnessId=harness_id)
print(f"Deleted harness: {harness_id}")

In [ ]:
control.delete_memory(memoryId=memory_id)
print(f"Deleted memory: {memory_id}")

In [ ]:
# IAM 역할 삭제

delete_harness_role()